# YOLOv8 — Xe tự hành miniature (320×240, low-light)

> **Class**: Cấu hình gán nhãn theo đúng thứ tự class

| ID | Class | Vật thể thực tế |
|----|-------|-----------------|
| 0 | `stop_sign` | Biển STOP bát giác đỏ thu nhỏ |
| 1 | `no_entry` | Biển cấm (tròn đỏ, vạch trắng) |
| 2 | `red_light` | LED đỏ trên cột đèn |
| 3 | `yellow_light` | LED vàng |
| 4 | `green_light` | LED xanh |
| 5 | `person` | Biển/hình người đi bộ (zebra crossing) |

> **Dataset**: ảnh 320×240, tối, chụp từ camera xe.  
```
/dataset/
├── images/
│   ├── train/  *.jpg
│   └── val/    *.jpg
└── labels/                   
    ├── train/  *.txt
    └── val/    *.txt

## ⚙️ Cell 1 — Cấu hình (chỉnh tại đây)

In [ ]:
from pathlib import Path

# ─────────────────────────────────────────────────────────────────
#  DATASET 
# ─────────────────────────────────────────────────────────────────
DATASET_INPUT = "/kaggle/input/datasets/buiminhphuongx/dataset/dataset"

# ─────────────────────────────────────────────────────────────────
#  MODEL
# ─────────────────────────────────────────────────────────────────
MODEL      = "yolov8n"    # n=nano (khuyên dùng cho RPi/Jetson)
                          # đổi sang "yolov8s" nếu GPU đủ mạnh

# ─────────────────────────────────────────────────────────────────
#  TRAINING 
# ─────────────────────────────────────────────────────────────────
EPOCHS     = 100
IMG_SIZE   = 320          
BATCH      = 32           
PATIENCE   = 25           # early-stop sau 25 epoch plateau

# ─────────────────────────────────────────────────────────────────
#  CLASSES
# ─────────────────────────────────────────────────────────────────
CLASSES = ["stop_sign", "no_entry", "red_light", "yellow_light", "green_light", "person"]

# ─────────────────────────────────────────────────────────────────
#  PATHS 
# ─────────────────────────────────────────────────────────────────
WORK       = Path("/kaggle/working")
DATASET    = WORK / "dataset"
RUNS       = WORK / "runs"
OUT        = WORK / "output"
OUT.mkdir(parents=True, exist_ok=True)

print(f"Model    : {MODEL}")
print(f"ImgSize  : {IMG_SIZE}×{IMG_SIZE}")
print(f"Epochs   : {EPOCHS}  |  Patience: {PATIENCE}")
print(f"Batch    : {BATCH}")
print(f"Classes  : {CLASSES}")
print(f"Dataset  : {DATASET_INPUT}")


## 1. Cài đặt

In [ ]:
%%capture
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install",
                "ultralytics>=8.2", "onnx", "onnxsim", "-q"], check=True)

import torch, cv2, ultralytics
print(f"ultralytics {ultralytics.__version__}  |  "
      f"torch {torch.__version__}  |  "
      f"opencv {cv2.__version__}")

if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f"GPU: {g.name}  |  VRAM: {g.total_memory/1e9:.1f} GB")
    DEVICE = "0"
else:
    print("Không có GPU — bật Accelerator trong Settings!")
    DEVICE = "cpu"


## 2. Chuẩn bị Dataset

In [ ]:
import shutil, yaml
from pathlib import Path

src = Path(DATASET_INPUT)
if not src.exists():
    raise FileNotFoundError(
        f"Không tìm thấy {src}\n"
        "→ Trên Kaggle: nhấn '+ Add data' → chọn dataset của bạn"
    )

# Copy sang /kaggle/working để tránh read-only
if DATASET.exists():
    shutil.rmtree(DATASET)
shutil.copytree(str(src), str(DATASET))

# Kiểm tra & thống kê nhanh
print("Dataset structure:")
for split in ["train", "val"]:
    imgs = list((DATASET/"images"/split).glob("*.[jp][pn]g"))
    lbls = list((DATASET/"labels"/split).glob("*.txt"))

    # Đếm annotation theo class
    counts = [0] * len(CLASSES)
    for lf in lbls:
        for line in lf.read_text().splitlines():
            p = line.strip().split()
            if len(p) == 5:
                cid = int(p[0])
                if 0 <= cid < len(CLASSES):
                    counts[cid] += 1

    print(f"  [{split}]  {len(imgs)} ảnh  |  {sum(counts)} boxes")
    for i, (cls, cnt) in enumerate(zip(CLASSES, counts)):
        bar = "█" * min(cnt // max(sum(counts)//30, 1), 25)
        print(f"    [{i}] {cls:<14} {cnt:4d}  {bar}")

# Tạo dataset.yaml
yaml_path = DATASET / "dataset.yaml"
yaml.dump({
    "path"  : str(DATASET.resolve()),
    "train" : "images/train",
    "val"   : "images/val",
    "nc"    : len(CLASSES),
    "names" : CLASSES,
}, open(yaml_path, "w"), default_flow_style=False, allow_unicode=True)

print(f"\ndataset.yaml → {yaml_path}")


## 3. Xem trước ảnh & nhãn

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt, matplotlib.patches as mpatches

COLORS = ["#e74c3c","#c0392b","#e67e22","#f1c40f","#2ecc71","#3498db"]

def show_labels(img_path, lbl_path, ax):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    ax.imshow(img)
    ax.set_title(img_path.name, fontsize=8)
    ax.axis("off")
    if lbl_path.exists():
        for line in lbl_path.read_text().splitlines():
            p = line.strip().split()
            if len(p) != 5: continue
            cid, cx, cy, bw, bh = int(p[0]), *map(float, p[1:])
            x1 = (cx - bw/2)*w;  y1 = (cy - bh/2)*h
            col = COLORS[cid % len(COLORS)]
            ax.add_patch(mpatches.Rectangle((x1,y1), bw*w, bh*h,
                lw=2, edgecolor=col, facecolor="none"))
            ax.text(x1, max(0,y1-2), CLASSES[cid], fontsize=7,
                    color=col, fontweight="bold",
                    bbox=dict(boxstyle="round,pad=0.1", fc="white", alpha=0.7))

imgs = sorted((DATASET/"images"/"train").glob("*.[jp][pn]g"))[:8]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, ip in zip(axes.flat, imgs):
    show_labels(ip, DATASET/"labels"/"train"/(ip.stem+".txt"), ax)
for ax in axes.flat[len(imgs):]:
    ax.axis("off")
plt.suptitle("Train set — ảnh thực tế + annotation", fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()


## 4. Train YOLOv8

In [ ]:
from ultralytics import YOLO

model = YOLO(f"{MODEL}.pt")

results = model.train(
    data    = str(yaml_path),
    epochs  = EPOCHS,
    imgsz   = IMG_SIZE,
    batch   = BATCH,
    device  = DEVICE,
    project = str(RUNS),
    name    = "traffic",
    exist_ok= True,

    # ── Augmentation cho ảnh tối 320×240 ─────────────────────────────
    # Tăng sáng/contrast mạnh — ảnh đường đua rất tối
    hsv_v       = 0.6,       # jitter brightness ±60%
    hsv_s       = 0.5,
    hsv_h       = 0.01,      # hue nhẹ — màu LED nhạy cảm
    # Geometric nhẹ — camera cố định trên xe, không nghiêng nhiều
    degrees     = 3.0,
    translate   = 0.05,
    scale       = 0.4,       # object nhỏ → scale mạnh để học đa dạng size
    shear       = 1.0,
    perspective = 0.0003,
    flipud      = 0.0,       # không lật dọc
    fliplr      = 0.5,
    # Mosaic + mixup giúp học object nhỏ trong context phức tạp
    mosaic      = 1.0,
    mixup       = 0.05,      # nhẹ — tránh làm mờ LED nhỏ
    copy_paste  = 0.0,       # off — background đường đua đồng nhất

    # ── Optimizer ────────────────────────────────────────────────────
    optimizer    = "AdamW",
    lr0          = 0.001,
    lrf          = 0.005,    # lr_final thấp hơn — fine-tune object nhỏ
    weight_decay = 0.0005,
    warmup_epochs= 5,        # warmup dài hơn — dataset nhỏ cần ổn định

    # ── Loss ─────────────────────────────────────────────────────────
    box = 10.0,   # tăng box loss — cần bbox chính xác cho object nhỏ
    cls = 0.3,    # giảm cls loss — chỉ 6 class, ít nhầm lẫn
    dfl = 1.5,

    # ── Misc ─────────────────────────────────────────────────────────
    patience    = PATIENCE,
    save_period = 20,
    plots       = True,
    verbose     = False,    # tắt spam log từng batch
)

# Tìm best.pt
best_pt = RUNS / "traffic" / "weights" / "best.pt"
if not best_pt.exists():
    cands = sorted(RUNS.rglob("best.pt"), key=lambda p: p.stat().st_mtime)
    best_pt = cands[-1] if cands else None

print(f"\n{'v' if best_pt and best_pt.exists() else 'x'} "
      f"Best weights: {best_pt}")


## 5. Kết quả & Biểu đồ

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

# ── Metrics validation ────────────────────────────────────────
val_model = YOLO(str(best_pt))
m = val_model.val(data=str(yaml_path), imgsz=IMG_SIZE, device=DEVICE, verbose=False)

print("┌──────────────────────────────┐")
print("│      VALIDATION METRICS      │")
print("├──────────────────────────────┤")
print(f"│  mAP@0.5      : {m.box.map50:.4f}        │")
print(f"│  mAP@0.5:0.95 : {m.box.map:.4f}        │")
print(f"│  Precision    : {m.box.mp:.4f}        │")
print(f"│  Recall       : {m.box.mr:.4f}        │")
print("├──────────────────────────────┤")
for i, cls in enumerate(CLASSES):
    try:
        ap = m.box.ap50[i]
        bar = "█"*int(ap*20)
        print(f"│  [{i}]{cls:<13}: {ap:.3f} {bar:<20}│")
    except: pass
print("└──────────────────────────────┘")

# ── Training curves ────────────────────────────────────────────
csv_path = RUNS / "traffic" / "results.csv"
if csv_path.exists():
    df = pd.read_csv(csv_path); df.columns = df.columns.str.strip()
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    pairs = [
        ("train/box_loss", "val/box_loss",      "Box Loss",    axes[0,0]),
        ("train/cls_loss", "val/cls_loss",      "Class Loss",  axes[0,1]),
        ("train/dfl_loss", "val/dfl_loss",      "DFL Loss",    axes[0,2]),
        ("metrics/precision(B)", None,           "Precision",   axes[1,0]),
        ("metrics/recall(B)",    None,           "Recall",      axes[1,1]),
        ("metrics/mAP50(B)",     None,           "mAP@0.5",     axes[1,2]),
    ]
    for tc, vc, title, ax in pairs:
        if tc in df: ax.plot(df[tc], label="train", lw=1.8, color="#3498db")
        if vc and vc in df: ax.plot(df[vc], label="val", lw=1.8, ls="--", color="#e74c3c")
        ax.set_title(title, fontweight="bold"); ax.legend(); ax.grid(alpha=0.3)
    plt.suptitle("Training Curves", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(str(OUT/"training_curves.png"), dpi=120, bbox_inches="tight")
    plt.show()

# ── Confusion matrix & PR curve ────────────────────────────────
run_dir = RUNS / "traffic"
for fname, title in [("confusion_matrix_normalized.png","Confusion Matrix"),
                      ("PR_curve.png","PR Curve")]:
    p = run_dir / fname
    if p.exists():
        fig, ax = plt.subplots(figsize=(8,6))
        ax.imshow(plt.imread(str(p))); ax.axis("off")
        ax.set_title(title, fontweight="bold")
        plt.tight_layout(); plt.show()


## 6. Inference trên ảnh thực tế

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt

infer = YOLO(str(best_pt))
val_imgs = sorted((DATASET/"images"/"val").glob("*.[jp][pn]g"))[:8]

if val_imgs:
    cols = 4; rows = (len(val_imgs)+cols-1)//cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, rows*4))
    axes = np.array(axes).flatten()
    for i, ip in enumerate(val_imgs):
        r = infer.predict(str(ip), imgsz=IMG_SIZE, conf=0.25, verbose=False)[0]
        plotted = cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)
        axes[i].imshow(plotted); axes[i].set_title(ip.name, fontsize=8)
        axes[i].axis("off")
    for ax in axes[len(val_imgs):]: ax.axis("off")
    plt.suptitle("Inference — Val Set", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(str(OUT/"inference_val.png"), dpi=100, bbox_inches="tight")
    plt.show()


## 7. Export ONNX & Tải về

In [ ]:
import shutil, zipfile

# ── Export ONNX ───────────────────────────────────────────────
exp = YOLO(str(best_pt))
onnx_src = Path(str(exp.export(
    format   = "onnx",
    imgsz    = IMG_SIZE,   # 320 — khớp với camera thực tế
    opset    = 12,         # tương thích OpenCV 4.x
    simplify = True,
    dynamic  = False,      # batch=1 fixed cho real-time
    half     = False,      # FP32 — tương thích RPi không GPU
)))
onnx_dst = OUT / "traffic_detector.onnx"
shutil.copy2(str(onnx_src), str(onnx_dst))
print(f"ONNX: {onnx_dst}  ({onnx_dst.stat().st_size/1e6:.2f} MB)")

# ── Copy best.pt ──────────────────────────────────────────────
shutil.copy2(str(best_pt), str(OUT/"best.pt"))

# ── Zip tất cả output ─────────────────────────────────────────
zip_path = WORK / "traffic_detector_results.zip"
with zipfile.ZipFile(str(zip_path), "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(str(OUT/"best.pt"),              "weights/best.pt")
    zf.write(str(onnx_dst),                   "weights/traffic_detector.onnx")
    zf.write(str(yaml_path),                  "dataset.yaml")
    for p in OUT.glob("*.png"):
        zf.write(str(p), f"plots/{p.name}")
    run_dir = RUNS/"traffic"
    for p in run_dir.glob("*.png"):
        zf.write(str(p), f"plots/{p.name}")
    csv_path = run_dir/"results.csv"
    if csv_path.exists():
        zf.write(str(csv_path), "results.csv")

print(f"\n{zip_path}  ({zip_path.stat().st_size/1e6:.1f} MB)")
print("   Kaggle → Output tab → traffic_detector_results.zip → Download")


## 8. Dùng ONNX trên xe tự hành

```python
# cv_lane_driver — tích hợp detector
import cv2, numpy as np

CLASSES    = ["stop_sign","no_entry","red_light","yellow_light","green_light","person"]
CONF_THRESH = 0.45
NMS_THRESH  = 0.40
IMG_SIZE    = 320          # phải khớp với lúc export

net = cv2.dnn.readNetFromONNX("traffic_detector.onnx")
net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)

def detect(frame):
    """frame: BGR numpy (bất kỳ size) → list[(cls_id, conf, x1,y1,x2,y2)]"""
    h, w = frame.shape[:2]
    blob = cv2.dnn.blobFromImage(frame, 1/255.0, (IMG_SIZE, IMG_SIZE),
                                  swapRB=True, crop=False)
    net.setInput(blob)
    out = net.forward()[0]           # shape (1, 4+nc, 8400) — YOLOv8 head

    # Transpose sang (8400, 4+nc)
    out = out.T
    boxes, confs, class_ids = [], [], []
    sx, sy = w/IMG_SIZE, h/IMG_SIZE

    for row in out:
        scores = row[4:]
        cid = int(np.argmax(scores))
        conf = float(scores[cid])
        if conf < CONF_THRESH:
            continue
        cx, cy, bw, bh = row[:4]
        x1 = int((cx - bw/2) * sx);  y1 = int((cy - bh/2) * sy)
        x2 = int((cx + bw/2) * sx);  y2 = int((cy + bh/2) * sy)
        boxes.append([x1, y1, x2-x1, y2-y1])
        confs.append(conf); class_ids.append(cid)

    idxs = cv2.dnn.NMSBoxes(boxes, confs, CONF_THRESH, NMS_THRESH)
    results = []
    for i in (idxs.flatten() if len(idxs) else []):
        x,y,w_,h_ = boxes[i]
        results.append((class_ids[i], confs[i], x, y, x+w_, y+h_))
    return results

# ── Vòng lặp chính ────────────────────────────────────────────────────────
cap = cv2.VideoCapture(0)
while True:
    ret, frame = cap.read()
    if not ret: break
    for cid, conf, x1, y1, x2, y2 in detect(frame):
        label = f"{CLASSES[cid]} {conf:.2f}"
        cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)
        cv2.putText(frame, label, (x1, y1-6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)

    # ── Hành động theo detection ──────────────────────────────────────────
    # if any(cid == 2 for cid,*_ in detections):   # red_light
    #     car.stop()
    # elif any(cid == 0 for cid,*_ in detections): # stop_sign
    #     car.stop(duration=2.0)

    cv2.imshow("traffic_detector", frame)
    if cv2.waitKey(1) == 27: break

cap.release(); cv2.destroyAllWindows()
```

### Ghi chú siêu tham số đã chọn
| Tham số | Giá trị | Lý do |
|---------|---------|-------|
| `IMG_SIZE` | 320 | Khớp ảnh thực tế, giảm latency trên RPi |
| `hsv_v` | 0.6 | Ảnh tối — augment brightness mạnh |
| `hsv_h` | 0.01 | LED có màu cụ thể — không jitter hue nhiều |
| `box` loss | 10.0 | Object nhỏ cần bbox chính xác |
| `cls` loss | 0.3 | Chỉ 6 class, ít nhầm lẫn |
| `mosaic` | 1.0 | Hiệu quả nhất cho small object |
| `mixup` | 0.05 | Nhẹ — tránh làm nhòe LED |
| `copy_paste` | 0.0 | Background đồng nhất → không cần |
| `warmup` | 5 epochs | Dataset nhỏ cần warm-up dài hơn |
